# Sequência de Fibonacci

A sequência de Fibonacci é definida por:

\[
F(0)=0
\]

\[
F(1)=1
\]

\[
F(n)=F(n-1)+F(n-2), \quad n \ge 2
\]

Este notebook mostra três versões em Cython:

1. recursiva ingênua;
2. recursiva com memoização;
3. iterativa bottom-up.

In [ ]:
%load_ext cython

## 1. Versão recursiva ingênua

A versão recursiva direta é simples, mas repete muitas chamadas.

Exemplo: `fib(5)` chama `fib(4)` e `fib(3)`. Depois, `fib(4)` também chama `fib(3)`. Assim, subproblemas iguais são resolvidos várias vezes.

### Análise assintótica

A recorrência aproximada é:

\[
T(n)=T(n-1)+T(n-2)+\Theta(1)
\]

O crescimento é exponencial, aproximadamente **Θ(φⁿ)**, onde `φ ≈ 1,618`.

In [ ]:
%%cython
# cython: boundscheck=False, wraparound=False, cdivision=True

cpdef long fib_recursivo(int n):
    if n <= 1:
        return n
    return fib_recursivo(n - 1) + fib_recursivo(n - 2)

In [ ]:
for n in range(11):
    print(n, fib_recursivo(n))

## 2. Memoização — top-down

A memoização guarda resultados já calculados. Assim, cada `F(k)` é calculado apenas uma vez.

### Análise assintótica

Existem apenas `n + 1` subproblemas distintos: `F(0), F(1), ..., F(n)`.

- tempo: **Θ(n)**;
- espaço: **Θ(n)** para o vetor de memoização e para a pilha recursiva.

In [ ]:
%%cython
# cython: boundscheck=False, wraparound=False, cdivision=True

cdef long _fib_memo(int n, list memo):
    if memo[n] != -1:
        return <long>memo[n]
    memo[n] = _fib_memo(n - 1, memo) + _fib_memo(n - 2, memo)
    return <long>memo[n]

def fib_memoizacao(int n):
    cdef list memo = [-1] * (n + 1)
    if n >= 0:
        memo[0] = 0
    if n >= 1:
        memo[1] = 1
    return _fib_memo(n, memo)

In [ ]:
for n in range(14):
    print(n, fib_memoizacao(n))

## 3. Programação dinâmica bottom-up

A versão iterativa constrói a solução do menor subproblema para o maior.

### Análise assintótica

O laço executa de `2` até `n`, portanto:

- tempo: **Θ(n)**;
- espaço: **Θ(n)** se armazenarmos todos os valores.

Também é possível otimizar o espaço para **Θ(1)** usando apenas os dois últimos valores.

In [ ]:
%%cython
# cython: boundscheck=False, wraparound=False, cdivision=True

def fib_bottom_up(int n):
    cdef int i
    cdef list r

    if n <= 1:
        return n

    r = [0] * (n + 1)
    r[0] = 0
    r[1] = 1

    for i in range(2, n + 1):
        r[i] = <long>r[i - 1] + <long>r[i - 2]

    return r[n]

def fib_bottom_up_espaco_constante(int n):
    cdef int i
    cdef long anterior = 0
    cdef long atual = 1
    cdef long proximo

    if n <= 1:
        return n

    for i in range(2, n + 1):
        proximo = anterior + atual
        anterior = atual
        atual = proximo

    return atual

In [ ]:
for n in range(14):
    print(n, fib_bottom_up(n), fib_bottom_up_espaco_constante(n))

## Comparação conceitual

| Versão | Ideia | Tempo | Espaço |
|---|---|---:|---:|
| Recursiva ingênua | resolve subproblemas repetidos | Θ(φⁿ) | Θ(n) |
| Memoização | salva resultados já calculados | Θ(n) | Θ(n) |
| Bottom-up | constrói de `0` até `n` | Θ(n) | Θ(n) |
| Bottom-up otimizada | guarda só dois valores | Θ(n) | Θ(1) |

In [ ]:
import time

for n in [10, 20, 30]:
    print(f"
n={n}")
    for nome, f in [
        ("memoização", fib_memoizacao),
        ("bottom-up", fib_bottom_up),
        ("espaço constante", fib_bottom_up_espaco_constante),
    ]:
        ini = time.perf_counter()
        r = f(n)
        fim = time.perf_counter()
        print(f"{nome:16s}: {r:10d} tempo={fim-ini:.8f}s")